### CEO Document Data

Generates and uploads all synthetic CEO-relevant document PDFs to Unity Catalog volumes:
- Legal complaints (employment, liability, vendor disputes)
- Regulatory documents (health permits, fire safety, zoning, FDA)
- Audit reports (financial, operational, food safety, supply chain)
- Consultancy reports (strategy, operations, AI transformation, workforce)

In [ ]:
%pip install --upgrade databricks-sdk fpdf2

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create UC schemas and volumes for CEO document types

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

for schema, volume in [
    ("legal_complaints", "documents"),
    ("regulatory", "documents"),
    ("audits", "reports"),
    ("consultancy", "reports"),
]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{schema}.{volume}")
    print(f"\u2705 Created {CATALOG}.{schema}.{volume}")

##### Create prompt registry schema

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.prompt_registry")
print(f"✅ Created schema {CATALOG}.prompt_registry")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.prompt_registry.prompts (
    name        STRING  NOT NULL COMMENT 'Unique prompt identifier, e.g. ceo_supervisor_system',
    version     INT     NOT NULL COMMENT 'Monotonically increasing version number',
    agent       STRING           COMMENT 'Agent or component this prompt belongs to',
    prompt      STRING  NOT NULL COMMENT 'Full prompt text',
    updated_at  TIMESTAMP        COMMENT 'Last modification timestamp',
    updated_by  STRING           COMMENT 'User or service principal that wrote this version'
)
USING DELTA
COMMENT 'Centralised prompt registry — one row per (name, version) pair'
""")
print(f"✅ Created table {CATALOG}.prompt_registry.prompts")

##### Run generators to produce PDFs locally, then upload to volumes

In [ ]:
import subprocess
import sys
import os

generators = [
    ("Legal Complaints",  "../data/legal_complaints/generate_legal_complaints.py"),
    ("Regulatory Docs",   "../data/regulatory/generate_regulatory_docs.py"),
    ("Audit Reports",     "../data/audits/generate_audit_reports.py"),
    ("Consultancy Reports", "../data/consultancy/generate_consultancy_reports.py"),
]

for name, script_path in generators:
    abs_path = os.path.abspath(script_path)
    print(f"Generating {name}...")
    result = subprocess.run(
        [sys.executable, abs_path],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}")
        raise RuntimeError(f"Generator failed for {name}: {result.stderr}")
    print(result.stdout)
    print(f"\u2705 {name} complete")

##### Upload PDFs to Unity Catalog volumes

In [ ]:
import glob

uploads = [
    ("../data/legal_complaints/pdfs", f"/Volumes/{CATALOG}/legal_complaints/documents"),
    ("../data/regulatory/pdfs",       f"/Volumes/{CATALOG}/regulatory/documents"),
    ("../data/audits/pdfs",           f"/Volumes/{CATALOG}/audits/reports"),
    ("../data/consultancy/pdfs",      f"/Volumes/{CATALOG}/consultancy/reports"),
]

for src_dir, vol_path in uploads:
    abs_src = os.path.abspath(src_dir)
    pdf_files = glob.glob(os.path.join(abs_src, "*.pdf"))
    print(f"Uploading {len(pdf_files)} PDFs to {vol_path}")
    for pdf_file in pdf_files:
        filename = os.path.basename(pdf_file)
        with open(pdf_file, "rb") as src:
            with open(f"{vol_path}/{filename}", "wb") as dst:
                dst.write(src.read())
        print(f"  Uploaded: {filename}")
    print(f"\u2705 Uploaded {len(pdf_files)} PDFs to {vol_path}")

##### Register completion in uc_state

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

print("\u2705 CEO Document Data stage complete")